# Notebook 3: reinforcement learning in brief

In notebooks 1–2 every example came with the **right answer** (the future flow). To *control* the plant there is no right answer to copy, only a **reward** that says how good the result was. That is reinforcement learning (RL).

We build a small, fast version of the project's control problem, so that everything trains in seconds:

| Step | What | Repository |
|---|---|---|
| 1 | an **environment** with `reset()` and `step()` | `src/wwtp/rl/env.py` (`AerationEnv`) |
| 2 | the **expert**: an incremental PID | `src/wwtp/baselines/pid.py` |
| 3 | **imitation**: copy the expert with a network (supervised!) | `src/wwtp/sacil/sacil.py` (`pretrain_bc`) |
| 4 | **Q-learning**: learn from reward alone | idea behind `rl/ddpg.py`, `sacil/sacil.py` |
| 5 | compare on days never used for training | `experiments/33_paper1_study.py` |
| 6 | change the reward and see what the agent does | `src/wwtp/rl/rewards.py` |


---
**New to notebooks? Read this first.**

- A notebook is a page made of **cells**. Grey cells hold Python code; white cells (like this one) hold text.
- Click a code cell and press **Shift + Enter**: the code runs and its result appears right under it. Then the next cell is selected.
- Run the cells **in order, from the top**. A cell can use things made by the cells above it.
- If something goes wrong: menu **Kernel → Restart Kernel and Run All Cells** starts everything again from a clean state.
- You cannot break anything. Change a number, run the cell again, and see what happens: that is how you learn.

> **In plain words:** think of teaching a dog a trick. You can't explain the trick in words; you can only give a **treat** when it does something good. The dog tries things, remembers what earned treats, and slowly gets better. That is **reinforcement learning**: the *agent* (the dog) takes *actions* in an *environment* (the world), and gets *rewards* (treats).

**Words used in this notebook**

| word | meaning |
|---|---|
| **agent** | the learner that decides what to do (here: how much air to blow) |
| **environment** | the world the agent acts in (here: a tank of water with bacteria) |
| **state** | what the agent can see right now (oxygen level, air setting, load) |
| **action** | what the agent does (turn the air up or down) |
| **reward** | a number after each action: high = good, low = bad |
| **episode** | one full try from start to end (here: one day) |
| **policy** | the agent's rule: "in this state, do this action" |
| **exploration** | trying something new to find out if it is better |
| **Q-value** | the agent's guess of how much reward it will collect if it takes this action now |
| **PID** | a classic controller from engineering, no learning: "too low? more air; too high? less air" |

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
DATA = Path("../data"); DATA.mkdir(exist_ok=True)
REV = "5dadb209a5a34ecb0e2f83a2e79c060bf3470582"
URL = "https://raw.githubusercontent.com/davidfernand/BSM1_in_Python/{rev}/code/Influent%20files/{name}"


def influent(name):
    """The official influent file (downloaded in notebook 1, or now)."""
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(URL.format(rev=REV, name=name), path)
    d = pd.read_excel(path).astype(float)
    d.columns = ["t", "S_I", "S_S", "X_I", "X_S", "X_BH", "X_BA", "X_P", "S_O", "S_NO", "S_NH", "S_ND", "X_ND", "S_ALK", "Q"]
    return d.iloc[:-1]                                   # drop the repeated last row


# The bacteria's oxygen demand follows the pollution load arriving: flow × (ammonium + some COD).
LOAD = {}
for weather, name in (("dry", "Inf_dry_2006.xlsx"), ("rain", "Inf_rain_2006.xlsx")):
    d = influent(name)
    load = (d.Q * (d.S_NH + 0.2 * d.S_S)).values
    LOAD[weather] = np.repeat(load / load.mean(), 3)    # 15-min samples → 5-min steps (held)

plt.plot(np.arange(len(LOAD["dry"])) / 288, LOAD["dry"], lw=1, label="dry")
plt.plot(np.arange(len(LOAD["rain"])) / 288, LOAD["rain"], lw=1, label="rain")
plt.xlabel("day"); plt.ylabel("load / mean"); plt.title("oxygen demand follows the incoming load"); plt.legend()

## 1. The environment

One aerated tank (like tank 5 of BSM1). Every 5 minutes the controller changes the aeration KLa. The oxygen then follows a simple mass balance:

**change of oxygen = KLa × (8 − oxygen)  −  OUR(t)**

(first part: air brings oxygen in; second part: the bacteria use it up)

solved exactly over each 5-minute step. 8 g/m³ is oxygen saturation; the demand OUR follows the real load profile.

- **state** (what the agent sees): oxygen error `S_O − 2`, the previous error, the current aeration `KLa/240`, the current load
- **action**: change KLa by one of −20, −5, 0, +5, +20 per step (incremental, like the papers)
- **reward**: `−|S_O − 2|  −  w · KLa/240`: be at 2 mg/L, and don't waste air (the same trade-off as paper 2's reward, `rl/rewards.py:39`)
- **episode**: one day (288 steps)

The interface is the standard one, the same as `AerationEnv` in the repository: `reset()` → first state; `step(action)` → next state, reward, done, info.


> **In plain words:** the environment is a **game** with clear rules. `reset()` starts a new game; `step(action)` makes one move and tells you the new situation and the points you scored. Every RL agent in the project plays this kind of game, just against a much more detailed plant.

In [ ]:
class OxygenTank:
    ACTIONS = np.array([-20.0, -5.0, 0.0, 5.0, 20.0])
    STEPS_PER_DAY, DT = 288, 5 / 1440           # 5-minute steps, in days

    def __init__(self, weather="dry", day=0, energy_weight=0.1, noise=0.05, seed=0):
        self.load, self.day, self.w, self.noise = LOAD[weather], day, energy_weight, noise
        self.rng = np.random.default_rng(seed)

    def reset(self):
        self.k0, self.k = int(self.day * self.STEPS_PER_DAY), 0
        self.kla, self.S, self.e1 = 120.0, 2.0, 0.0
        return self.obs()

    def obs(self):
        load_now = self.load[self.k0 + min(self.k, self.STEPS_PER_DAY - 1)]
        return np.array([self.S - 2.0, self.e1, self.kla / 240.0, load_now])

    def step(self, dkla):
        self.e1 = self.S - 2.0
        self.kla = float(np.clip(self.kla + dkla, 0.0, 240.0))
        our = 700.0 * self.load[self.k0 + self.k] * (1 + self.noise * self.rng.normal())
        if self.kla > 1e-6:                                         # exact solution over the step
            s_eq = 8.0 - our / self.kla
            self.S = s_eq + (self.S - s_eq) * np.exp(-self.kla * self.DT)
        else:
            self.S = self.S - our * self.DT
        self.S = max(self.S, 0.0)
        self.k += 1
        reward = -abs(self.S - 2.0) - self.w * self.kla / 240.0
        done = self.k >= self.STEPS_PER_DAY
        return self.obs(), reward, done, {"S": self.S, "kla": self.kla}


def run(env, policy):
    """One episode; returns the total reward and the traces."""
    o, done, R, S, K = env.reset(), False, 0.0, [], []
    while not done:
        o, r, done, info = env.step(policy(o))
        R += r; S.append(info["S"]); K.append(info["kla"])
    S, K = np.array(S), np.array(K)
    return {"return": R, "oxygen error": np.abs(S - 2).mean(), "mean KLa": K.mean(), "S": S, "K": K}


# try two silly policies first: always the same air, and random
for name, pol in (("constant air", lambda o: 0.0),
                  ("random", lambda o, rng=np.random.default_rng(0): rng.choice(OxygenTank.ACTIONS))):
    r = run(OxygenTank("dry", day=10), pol)
    print(f"{name:13s} return {r['return']:8.1f}   mean |S_O − 2| {r['oxygen error']:.3f} mg/L")

## 2. The expert: an incremental PID

`Δu = Kp (e − e_prev) + Ki e` (the velocity form of `baselines/pid.py`). It needs no data and no training: it is the baseline to beat, and the "expert knowledge" of paper 1.


> **In plain words:** before any learning, use what engineers already know. A PID is a thermostat for oxygen: if the oxygen is too low, add air; if too high, remove air; the further off, the bigger the change. No data, no training, and it works. This is the bar any learner must clear.

In [ ]:
def pid_policy(o, kp=80.0, ki=25.0):
    e, e1 = -o[0], -o[1]                        # the PID uses r − s
    return float(np.clip(kp * (e - e1) + ki * e, -20, 20))


res_pid = run(OxygenTank("dry", day=10), pid_policy)
print(f"PID  return {res_pid['return']:.1f}   mean |S_O − 2| {res_pid['oxygen error']:.3f} mg/L   mean KLa {res_pid['mean KLa']:.0f}")

fig, ax = plt.subplots(2, 1, figsize=(11, 4.6), sharex=True)
h = np.arange(288) / 12
ax[0].plot(h, res_pid["S"]); ax[0].axhline(2, color="k", ls="--", lw=1); ax[0].set_ylabel("oxygen [mg/L]")
ax[1].plot(h, res_pid["K"], color="tab:orange"); ax[1].set_ylabel("KLa [1/d]"); ax[1].set_xlabel("hour of day 10")
ax[0].set_title("PID on dry day 10")

## 3. Imitation: learn to copy the expert (this is supervised learning)

Run the PID on the **training days (dry days 0–6)**, record every (state, action) pair, and train a small network to predict the action from the state. The project's controller starts exactly this way (`sacil.py:279`, `pretrain_bc`) before any reinforcement learning, so it never begins from random actions on the plant.


> **In plain words:** learning by **watching a teacher**. We record what the PID did in many situations and train a network to copy it. This is ordinary supervised learning: the question is the situation, the answer is the teacher's move. A copy can be as good as the teacher, never better.

In [ ]:
obs_list, act_list = [], []
for day in range(7):
    env = OxygenTank("dry", day=day, seed=day)
    o, done = env.reset(), False
    while not done:
        a = pid_policy(o)
        obs_list.append(o); act_list.append(a / 20.0)          # scale the action to [−1, 1]
        o, _, done, _ = env.step(a)
Xd = torch.tensor(np.array(obs_list), dtype=torch.float32)
yd = torch.tensor(np.array(act_list), dtype=torch.float32)[:, None]
print("demonstrations:", Xd.shape, "→", yd.shape)

torch.manual_seed(0)
bc = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh())
opt = torch.optim.Adam(bc.parameters(), lr=3e-3)
for it in range(3000):
    idx = torch.randint(0, len(Xd), (256,))
    loss = ((bc(Xd[idx]) - yd[idx]) ** 2).mean()               # plain regression, as in pretrain_bc
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 1000 == 0 or it == 2999:
        print(f"step {it:4d}  imitation loss {loss.item():.5f}")


def bc_policy(o):
    with torch.no_grad():
        return 20.0 * float(bc(torch.tensor(o, dtype=torch.float32)))

res_bc = run(OxygenTank("dry", day=10), bc_policy)
print(f"\nimitation  return {res_bc['return']:.1f}   mean |S_O − 2| {res_bc['oxygen error']:.3f}   (PID: {res_pid['oxygen error']:.3f})")

The copy is as good as the expert, and **no better**: imitation can't exceed its teacher. To improve, the agent needs the reward.

## 4. Q-learning: learn from reward alone

The agent keeps a table `Q[state, action]`: "how much reward will I collect from here if I take this action and act well afterwards". After every step it moves its estimate towards what it just experienced (the **Bellman** update):

**new Q(s, a) = old Q(s, a) + α × ( r + γ × best Q of the next state − old Q(s, a) )**

- **α** learning rate, **γ** how much future reward counts;
- **exploration**: with probability ε take a random action (ε falls from 1 to 0.05), otherwise the best known one;
- the continuous state is cut into bins, so that a table can hold it.

DDPG and SAC (the repository's agents) do the same thing with **networks** instead of a table: the critic network replaces `Q`, the actor network replaces `argmax` (`rl/ddpg.py:214`).


> **In plain words:** the agent keeps a big **notebook**: for every situation and every possible move, "how good was this, as far as I know?". After each move it updates that page a little, using the treat it just got plus how good the next situation looks. At first it tries random moves (**exploration**, like trying a new restaurant); later it mostly picks the best-known move (**exploitation**, like going back to your favourite).

In [ ]:
E_BINS = np.array([-0.5, -0.25, -0.1, 0.1, 0.25, 0.5])      # oxygen error
K_BINS = np.linspace(0, 1, 17)[1:-1]                         # aeration level
L_BINS = np.array([0.6, 0.8, 1.0, 1.2, 1.4])                 # load


def state(o):
    return np.digitize(o[0], E_BINS), np.digitize(o[2], K_BINS), np.digitize(o[3], L_BINS)


def greedy(Q):
    return lambda o: OxygenTank.ACTIONS[int(np.argmax(Q[state(o)]))]


def q_learning(episodes=1500, alpha=0.2, gamma=0.9, energy_weight=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((len(E_BINS) + 1, len(K_BINS) + 1, len(L_BINS) + 1, len(OxygenTank.ACTIONS)))
    curve, best = [], (-np.inf, None, 0)
    for ep in range(episodes):
        eps = max(0.05, 1 - ep / (0.6 * episodes))
        env = OxygenTank("dry", day=int(rng.integers(0, 7)), energy_weight=energy_weight, seed=10_000 * seed + ep)
        o, done, R = env.reset(), False, 0.0
        s = state(o)
        while not done:
            a = int(rng.integers(len(OxygenTank.ACTIONS))) if rng.random() < eps else int(np.argmax(Q[s]))
            o, r, done, _ = env.step(OxygenTank.ACTIONS[a])
            s2 = state(o)
            target = r + (0.0 if done else gamma * Q[s2].max())
            Q[s + (a,)] += alpha * (target - Q[s + (a,)])          # the Bellman update
            s, R = s2, R + r
        curve.append(R)
        if ep % 100 == 99:                                      # check the greedy policy on a training day
            val = run(OxygenTank("dry", day=6, energy_weight=energy_weight), greedy(Q))["return"]
            if val > best[0]:
                best = (val, Q.copy(), ep)                      # keep the best, as the repository does
    return best[1], np.array(curve), best[2]


Qtab, curve, kept = q_learning(seed=0)
print("table size:", Qtab.shape, "=", Qtab.size, "numbers   kept the policy of episode", kept)
plt.plot(curve, lw=.6, alpha=.5, label="each training episode")
plt.plot(np.convolve(curve, np.ones(50) / 50, mode="valid"), lw=2, label="moving average")
pid_train = np.mean([run(OxygenTank("dry", day=d, seed=d), pid_policy)["return"] for d in range(7)])
plt.axhline(pid_train, color="k", ls="--", label="PID, average over the training days")
plt.xlabel("episode (one day)"); plt.ylabel("return"); plt.ylim(-400, 0); plt.legend(); plt.title("Q-learning: return per episode")

Early on the agent acts randomly (ε ≈ 1) and the return is terrible: on a real plant those would be days of bad effluent. This is why the project starts from the expert (imitation) and keeps a safety gate.

### A stronger baseline: the PID plus engineering knowledge

The Q-learning agent sees the load; the PID does not. To be fair, give the PID the same information the engineer's way. From the mass balance, holding 2 mg/L at demand OUR needs `KLa = OUR / (8 − 2)`. So add the change of that value to the PID's move (a **feedforward**). This mirrors the project, where a tuned PID with a learned daily feedforward (ILC) was the strongest baseline.

In [ ]:
def make_pid_ff(kp=80.0, ki=25.0):
    last = {"ff": None}

    def policy(o):
        ff = 700.0 * o[3] / 6.0                                 # KLa that holds 2 mg/L at this load
        dff = 0.0 if last["ff"] is None else ff - last["ff"]
        last["ff"] = ff
        return float(np.clip(pid_policy(o, kp, ki) + dff, -20, 20))
    return policy

## 5. Compare on days never used for training, over several seeds

Training used dry days 0–6. We test on **dry day 10** and on **rain day 9** (the rain event: a weather the agent never saw). RL results depend on the random seed, so we train three agents.


> **In plain words:** a good student can solve **new** problems, not just the ones from class. So we test on days the agent never trained on, including a rainy day it has never seen. And because luck plays a role in learning, we train three agents with different random seeds and look at all of them.

In [ ]:
agents = [Qtab] + [q_learning(seed=s)[0] for s in (1, 2)]
tests = {"dry day 10": ("dry", 10), "rain day 9 (unseen weather)": ("rain", 9)}
rows = []
for tname, (w, day) in tests.items():
    for name, pol in (("PID (expert)", pid_policy), ("imitation", bc_policy), ("PID + feedforward", make_pid_ff())):
        r = run(OxygenTank(w, day=day), pol)
        rows.append((tname, name, r["return"], r["oxygen error"], r["mean KLa"]))
    rs = [run(OxygenTank(w, day=day), greedy(Q)) for Q in agents]
    for i, r in enumerate(rs):
        rows.append((tname, f"Q-learning seed {i}", r["return"], r["oxygen error"], r["mean KLa"]))
table = pd.DataFrame(rows, columns=["test", "controller", "return", "mean |S_O − 2|", "mean KLa"]).round(3)
table

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True)
for name, pol in (("PID", pid_policy), ("PID + feedforward", make_pid_ff()), ("Q-learning seed 0", greedy(agents[0]))):
    r = run(OxygenTank("rain", day=9), pol)
    ax[0].plot(h, r["S"], lw=1.2, label=name); ax[1].plot(h, r["K"], lw=1.2, label=name)
ax[0].axhline(2, color="k", ls="--", lw=1); ax[0].set_ylabel("oxygen [mg/L]"); ax[0].legend(ncol=3)
ax[1].set_ylabel("KLa [1/d]"); ax[1].set_xlabel("hour of rain day 9"); ax[0].set_title("rain day 9: never seen in training")

**Discuss:** Q-learning beats the plain PID, on every seed and on the unseen rain day. But the PID with one line of physics (the feedforward) does as well, without any training, exploration or bad days. The project found the same with far bigger agents: the learned controller beat the paper's PID, and a well-tuned PID with a daily feedforward matched it. **Always ask: better than what?**

## 6. The reward is the specification

The agent optimises **exactly** what the reward says. Make air fifty times more expensive (`energy_weight = 5`) and retrain:


> **In plain words:** the dog does exactly what earns treats, not what you had in mind. If you reward "quiet", it may learn to hide under the bed. Here, if air is too expensive in the reward, the agent simply turns the air off and the bacteria suffocate. **Writing the reward is writing the rules: write them carefully.**

In [ ]:
Q_cheap, _, _ = q_learning(energy_weight=5.0, seed=0)
for name, pol in (("PID", pid_policy), ("Q-learning, w = 0.1", greedy(agents[0])), ("Q-learning, w = 5", greedy(Q_cheap))):
    r = run(OxygenTank("dry", day=10, energy_weight=5.0), pol)
    print(f"{name:20s} mean oxygen {r['S'].mean():.2f} mg/L   mean KLa {r['mean KLa']:5.1f}   time below 1.5 mg/L {np.mean(r['S'] < 1.5):.0%}")

With expensive air the agent almost switches the air off and lets the bacteria go without oxygen. Each step it loses at most 2 for the oxygen error, but saves more than that in air. Nothing in the reward says the oxygen must stay above 1.5 mg/L, so it doesn't. Try `energy_weight` 2 and 3: where does the behaviour flip? This is what happened to paper 2's DDPG: it saved 9 % energy and broke the ammonium limit 54–65 % of the time. The repository's answer (`rl/panda_rl.py`) turns limits into **constraints** with their own budget instead of guessing weights.

### What you just did

- wrote an RL **environment** (`reset`, `step`, reward), the interface every agent in the repository uses
- used a classical **expert** as the baseline
- **imitated** it with supervised learning, the first stage of the project's SAC-IL controller
- trained a **Q-learning** agent from reward alone, kept its best policy, tested it on unseen days over several seeds
- saw that the **reward is the specification**: the agent does what you wrote, not what you meant